# Séries historiques de population (INSEE)

Le RP 2023 publie les populations de 2017 et 2023 à la géographie du 1er janvier 2026, ainsi que la superficie et l'état civil.

In [6]:
# Préparer les chemins, lire les réglages et créer le dossier de sortie.
import os
from pathlib import Path
import sys

# Trouver le dossier partagé depuis le dépôt ou le dossier notebooks.
racine = Path.cwd()
while not (racine / "config.yaml").is_file() and racine.parent != racine:
    dossier_insee = racine / "insee_population_navettes"
    if (dossier_insee / "config.yaml").is_file():
        racine = dossier_insee
        break
    racine = racine.parent
if not (racine / "config.yaml").is_file():
    raise FileNotFoundError("Impossible de trouver le dossier insee_population_navettes.")
os.chdir(racine)
sys.path.insert(0, str(racine))

from src.collecte import lire_config

config = lire_config()
config["hors_ligne"] = os.environ.get("RAIL_HORS_LIGNE") == "1"
propre = Path(config["dossier_propre"])
propre.mkdir(parents=True, exist_ok=True)

## Collecte

L'instantané est archivé dans `data/raw/rail2026`. La collecte suivante relit le cache.

In [7]:
# Lire les s?ries INSEE brutes ; les premi?res lignes ne sont pas encore filtr?es.
from src.insee import get_population

brut = get_population(config)
brut.head(3)

INSEE population : 3,049,137 lignes brutes lues ; archive et empreinte contrôlées.


,GEO,GEO_OBJECT,FREQ,RP_MEASURE,OCS,OBS_STATUS,UNIT_MEASURE,TIME_PERIOD,OBS_VALUE
0,001,AAV2020,A,DWELLINGS,DW_MAIN,A,NR,1968,3446694.0
1,001,AAV2020,A,DWELLINGS,DW_SEC_DW_OCC,A,NR,1968,156421.0
2,001,AAV2020,A,DWELLINGS,DW_VAC,A,NR,1968,175385.0


## Nettoyage

On sélectionne les mesures utiles et leurs dates avant de pivoter. Les statuts INSEE de regroupement restent visibles. Ils permettent de repérer les comparaisons communales fragiles.

In [8]:
# Garder les communes et mesures utiles, puis r?unir les ann?es dans une ligne.
from src.insee import clean_population

table = clean_population(brut, config)
display(table.head())

Population propre : 34,746 communes métropolitaines, sans double compte des arrondissements.
  2017 : 0 valeurs absentes, 0 alertes de périmètre ; somme publiée 64,639,133.
  2023 : 0 valeurs absentes, 0 alertes de périmètre ; somme publiée 66,165,815.


,code_commune,pop_2017,statut_pop_2017,alerte_population_2017,pop_2023,statut_pop_2023,alerte_population_2023,superficie_km2,naissances_2017_2022,deces_2017_2022
0,01001,776.0,A,False,860.0,A,False,15.95,52.0,34.0
1,01002,248.0,A,False,270.0,A,False,9.15,12.0,11.0
2,01004,14035.0,A,False,15934.0,A,False,24.60,1336.0,822.0
3,01005,1689.0,A,False,1906.0,A,False,15.92,157.0,59.0
4,01006,111.0,A,False,115.0,A,False,5.88,4.0,6.0


## Contrôle et export

La clé ou le couple de clés identifie chaque observation. Le fichier propre garde ces types pour les jointures.

In [9]:
# V?rifier qu?il reste exactement une ligne par commune.
assert table["code_commune"].is_unique
table[["pop_2017", "pop_2023"]].sum()

pop_2017    64639133.0
pop_2023    66165815.0
dtype: float64

In [10]:
# Enregistrer la table nettoy?e pour les autres ?tapes du projet.
table.to_parquet(propre / "source_population.parquet", index=False)